In [1]:
import os

# Create app folder
os.makedirs(
    "app",
    exist_ok=True
)

# Create pages folder inside app
os.makedirs(
    "app/pages",
    exist_ok=True
)

print("✅ app and pages folders created")

✅ app and pages folders created


In [3]:
%%writefile app/pages/forecasting.py

print("Forecasting Page Ready ✅")

Writing app/pages/forecasting.py


In [4]:
%%writefile app/pages/forecasting.py

import streamlit as st

import pandas as pd

import numpy as np

import plotly.graph_objects as go

# ============================================================
# MOCK DATA HELPERS
# ============================================================

def load_top_skus():

    return [

        "SKU10105",

        "SKU10114",

        "SKU10100",

        "SKU10045",

        "SKU10108"

    ]

def load_mape(sku):

    return round(

        np.random.uniform(4.0, 6.0),

        2

    )

def load_forecast(sku, horizon):

    dates = pd.date_range(

        start="2026-01-01",

        periods=horizon

    )

    actual = np.random.randint(

        80,

        140,

        size=horizon

    )

    forecast = actual + np.random.normal(

        0,

        8,

        size=horizon

    )

    upper = forecast + 15

    lower = forecast - 15

    return pd.DataFrame({

        "ds": dates,

        "actual": actual,

        "yhat": forecast,

        "yhat_upper": upper,

        "yhat_lower": lower

    })

# ============================================================
# MAIN PAGE
# ============================================================

def render():

    st.title(

        "📈 Demand Forecasting"

    )

    # ========================================================
    # CONTROLS
    # ========================================================

    col1, col2, col3 = st.columns(3)

    sku = col1.selectbox(

        "Select SKU",

        load_top_skus()

    )

    horizon = col2.slider(

        "Forecast horizon (days)",

        7,

        90,

        30

    )

    promo = col3.slider(

        "Promo lift (%)",

        -20,

        50,

        0,

        help="Apply what-if demand adjustment"

    )

    # ========================================================
    # LOAD DATA
    # ========================================================

    fcst = load_forecast(

        sku,

        horizon

    )

    fcst['yhat_adj'] = (

        fcst['yhat']

        * (1 + promo / 100)

    )

    # ========================================================
    # PLOT
    # ========================================================

    fig = go.Figure()

    fig.add_trace(

        go.Scatter(

            x=fcst['ds'],

            y=fcst['actual'],

            name='Historical'

        )

    )

    fig.add_trace(

        go.Scatter(

            x=fcst['ds'],

            y=fcst['yhat'],

            name='Forecast'

        )

    )

    fig.add_trace(

        go.Scatter(

            x=fcst['ds'],

            y=fcst['yhat_upper'],

            line=dict(width=0),

            showlegend=False

        )

    )

    fig.add_trace(

        go.Scatter(

            x=fcst['ds'],

            y=fcst['yhat_lower'],

            fill='tonexty',

            name='95% CI',

            line=dict(width=0)

        )

    )

    # ========================================================
    # WHAT-IF ANALYSIS
    # ========================================================

    if promo != 0:

        fig.add_trace(

            go.Scatter(

                x=fcst['ds'],

                y=fcst['yhat_adj'],

                name=f'What-if ({promo:+}%)',

                line=dict(dash='dot')

            )

        )

    st.plotly_chart(

        fig,

        use_container_width=True

    )

    # ========================================================
    # METRICS
    # ========================================================

    m1, m2, m3 = st.columns(3)

    m1.metric(

        "MAPE",

        f"{load_mape(sku):.2f}%"

    )

    m2.metric(

        "Avg Daily Demand",

        f"{fcst['yhat'].mean():.0f} units"

    )

    m3.metric(

        "Total Forecast",

        f"{fcst['yhat'].sum():.0f} units"

    )

Overwriting app/pages/forecasting.py
